# Qwen Binary-Classifier Dataset Builder (wikidata)

Builds a balanced `(anchor_query, query, label)` pair dataset for the **binary-classification** fine-tuning baseline (consumed by `02_qwen_finetuning.ipynb`).

- Input  : `dataset/raw/jaccard_triplets_wikidata.json`
- Splits : `dataset/splits/splits_wikidata.json`   (same `train_idx` / `valid_idx` as SPARQLCE)
- Output : `dataset/qwen_binary/wikidata/{train,val}.json` + `DATASET_CARD.txt`

### Format
```
{ "anchor_query": str, "query": str, "label": 0 | 1 }
```

label=1 → semantically similar to anchor  
label=0 → dissimilar (hard negative from the triplet record, cross-anchor fill only if needed)

### Design notes
- Uses **exactly** SPARQLCE's train/val split → head-to-head comparable.
- Anchor-disjoint by construction (splits file guarantees this).
- Cross-anchor negatives drawn **only** from within the same split to avoid leakage.
- No TF-IDF hard mining here: the triplet record already contains Jaccard-mined hard negatives.

In [1]:
import os
import json
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

SEED               = 42
MAX_POS_PER_ANCHOR = 20
DATASET_NAME       = "wikidata"


def find_repo_root(start=None):
    """Walk up from `start` (default cwd) until a dir holding dataset/raw/ and dataset/splits/ is found."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "dataset" / "raw").is_dir() and (p / "dataset" / "splits").is_dir():
            return p
    raise FileNotFoundError(
        f"Could not locate repo root containing dataset/raw/ and dataset/splits/ starting from: {start}"
    )


env_root = os.environ.get("SPARQL_EMB_ROOT")
REPO_ROOT   = Path(env_root).resolve() if env_root else find_repo_root()
DATA_PATH   = REPO_ROOT / "dataset" / "raw"    / f"jaccard_triplets_{DATASET_NAME}.json"
SPLITS_PATH = REPO_ROOT / "dataset" / "splits" / f"splits_{DATASET_NAME}.json"

# Output lives NEXT TO this notebook (i.e. finetuning_llm/binary_classifier/qwen_finetune_dataset/<ds>/).
NOTEBOOK_DIR = Path.cwd().resolve()
OUTPUT_DIR   = NOTEBOOK_DIR / "../qwen_finetune_dataset" / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(),   f"Missing raw triplets: {DATA_PATH}"
assert SPLITS_PATH.exists(), f"Missing splits file : {SPLITS_PATH}"

rng = random.Random(SEED)
np.random.seed(SEED)

print(f"Repo root   : {REPO_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Triplets    : {DATA_PATH}")
print(f"Splits      : {SPLITS_PATH}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Max pos/anchor: {MAX_POS_PER_ANCHOR}")

Repo root   : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings
Notebook dir: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/wikidata
Triplets    : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/dataset/raw/jaccard_triplets_wikidata.json
Splits      : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/dataset/splits/splits_wikidata.json
Output dir  : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/wikidata/../qwen_finetune_dataset/wikidata
Max pos/anchor: 20


In [2]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

for required in ("train_idx", "valid_idx"):
    assert required in splits, f"splits file missing '{required}'; got keys {list(splits.keys())}"

train_idx = splits["train_idx"]
valid_idx = splits["valid_idx"]

n_pos_total = sum(len(r.get("positive_queries", [])) for r in raw)
n_neg_total = sum(len(r.get("negative_queries", [])) for r in raw)
print(f"Anchors total  : {len(raw)}")
print(f"Positives total: {n_pos_total}")
print(f"Negatives total: {n_neg_total}")
print(f"train_idx      : {len(train_idx)}")
print(f"valid_idx      : {len(valid_idx)}")

Anchors total  : 10735
Positives total: 60729
Negatives total: 214700
train_idx      : 8588
valid_idx      : 1073


In [3]:
def _get(rec, *keys):
    for k in keys:
        if k in rec:
            return rec[k]
    raise KeyError(f"None of {keys} found in {list(rec.keys())}")


def _as_text(item):
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        return _get(item, "query", "sparql", "sparqlQuery", "text").strip()
    raise TypeError(f"Unexpected query container type: {type(item)}")


def parse_records(records):
    """Normalise raw triplet records into {anchor, positives, negatives} dicts.

    - Deduplicates positives and negatives.
    - Removes any negative that appears as a positive for the same anchor (paranoid safety).
    - Drops records with zero positives.
    """
    parsed = []
    for rec in records:
        anchor   = _as_text(_get(rec, "anchor_query", "anchor"))
        pos_raw  = rec.get("positive_queries", rec.get("positives", []))
        neg_raw  = rec.get("negative_queries", rec.get("negatives", rec.get("hard_negatives", [])))

        positives = list(dict.fromkeys(_as_text(p) for p in pos_raw))
        pos_set   = set(positives)
        negatives = list(dict.fromkeys(
            n for n in (_as_text(x) for x in neg_raw)
            if n and n not in pos_set and n != anchor
        ))
        if not positives:
            continue
        parsed.append({"anchor": anchor, "positives": positives, "negatives": negatives})
    return parsed


train_records = parse_records([raw[i] for i in train_idx])
val_records   = parse_records([raw[i] for i in valid_idx])
print(f"Parsed train records: {len(train_records)}  (from {len(train_idx)} indices)")
print(f"Parsed val   records: {len(val_records)}  (from {len(valid_idx)} indices)")

Parsed train records: 8588  (from 8588 indices)
Parsed val   records: 1073  (from 1073 indices)


In [4]:
def build_balanced_pairs(parsed_records, seed, max_pos=MAX_POS_PER_ANCHOR):
    """Produce balanced (anchor, query, label) pairs for ONE split.

    Per anchor:
      1. n_pos = min(len(positives), max_pos)   (random sub-sample if capped)
      2. n_neg = n_pos   (hard negatives first, cross-anchor fill if needed)
      3. If negatives are still insufficient, reduce positives to match.

    Cross-anchor negative pool is restricted to positives of OTHER anchors
    within `parsed_records` — call this function separately per split to
    avoid train/val leakage.
    """
    local_rng = random.Random(seed)

    cross_pool = [(rec["anchor"], q) for rec in parsed_records for q in rec["positives"]]

    pairs = []
    stats = {"pos": 0, "neg_hard": 0, "neg_cross": 0, "reduced": 0}

    for rec in parsed_records:
        anchor    = rec["anchor"]
        positives = rec["positives"][:]
        negatives = rec["negatives"][:]
        pos_full  = set(positives)

        if len(positives) > max_pos:
            positives = local_rng.sample(positives, max_pos)
        n_target = len(positives)

        local_rng.shuffle(negatives)
        selected = []
        for n in negatives:
            if len(selected) >= n_target:
                break
            if n not in pos_full and n != anchor:
                selected.append((n, "hard"))

        deficit = n_target - len(selected)
        if deficit > 0:
            cand = list(dict.fromkeys(
                q for (a, q) in cross_pool
                if a != anchor and q not in pos_full and q != anchor
            ))
            local_rng.shuffle(cand)
            for q in cand[:deficit]:
                selected.append((q, "cross"))

        if len(selected) < n_target:
            positives = positives[:len(selected)]
            stats["reduced"] += 1

        n_final = len(positives)
        selected = selected[:n_final]

        for q in positives:
            pairs.append({"anchor_query": anchor, "query": q, "label": 1})
            stats["pos"] += 1
        for q, src in selected:
            pairs.append({"anchor_query": anchor, "query": q, "label": 0})
            stats["neg_hard" if src == "hard" else "neg_cross"] += 1

    return pairs, stats


train_pairs, train_stats = build_balanced_pairs(train_records, seed=SEED)
val_pairs,   val_stats   = build_balanced_pairs(val_records,   seed=SEED + 1)

print("Train stats:", train_stats)
print("Val   stats:", val_stats)

Train stats: {'pos': 48569, 'neg_hard': 48569, 'neg_cross': 0, 'reduced': 0}
Val   stats: {'pos': 6183, 'neg_hard': 6183, 'neg_cross': 0, 'reduced': 0}


In [5]:
df_train = pd.DataFrame(train_pairs).drop_duplicates(subset=["anchor_query", "query", "label"]).reset_index(drop=True)
df_val   = pd.DataFrame(val_pairs  ).drop_duplicates(subset=["anchor_query", "query", "label"]).reset_index(drop=True)

train_anchors = set(df_train["anchor_query"])
val_anchors   = set(df_val["anchor_query"])
anchor_overlap = train_anchors & val_anchors


def _split_summary(df, name):
    pos = int((df["label"] == 1).sum())
    neg = int((df["label"] == 0).sum())
    per = df.groupby("anchor_query").size()
    return {
        "name"                   : name,
        "pairs"                  : int(len(df)),
        "anchors"                : int(df["anchor_query"].nunique()),
        "pos"                    : pos,
        "neg"                    : neg,
        "mean_pairs_per_anchor"  : float(per.mean())   if len(per) else 0.0,
        "median_pairs_per_anchor": float(per.median()) if len(per) else 0.0,
    }


train_sum = _split_summary(df_train, "train")
val_sum   = _split_summary(df_val,   "val")

print(f"train : {train_sum}")
print(f"val   : {val_sum}")
print(f"anchor overlap (train \u2229 val): {len(anchor_overlap)}   "
      f"{'OK (anchor-disjoint)' if not anchor_overlap else 'LEAKAGE — check splits file'}")

train : {'name': 'train', 'pairs': 97138, 'anchors': 8588, 'pos': 48569, 'neg': 48569, 'mean_pairs_per_anchor': 11.310898928737773, 'median_pairs_per_anchor': 4.0}
val   : {'name': 'val', 'pairs': 12366, 'anchors': 1073, 'pos': 6183, 'neg': 6183, 'mean_pairs_per_anchor': 11.524697110904008, 'median_pairs_per_anchor': 4.0}
anchor overlap (train ∩ val): 0   OK (anchor-disjoint)


In [6]:
COLS = ["anchor_query", "query", "label"]

df_train[COLS].to_json(OUTPUT_DIR / "train.json", orient="records", indent=2, force_ascii=False)
df_val  [COLS].to_json(OUTPUT_DIR / "val.json",   orient="records", indent=2, force_ascii=False)

card = f"""
================================================================
  Qwen Binary-Classifier Dataset  —  {DATASET_NAME.upper()}
================================================================
Format       : anchor_query | query | label (0 or 1)
Split source : {SPLITS_PATH.name}  (train_idx / valid_idx)
Raw source   : {DATA_PATH.name}

Train : {train_sum['pairs']} pairs  |  {train_sum['anchors']} anchors
         pos={train_sum['pos']}  neg={train_sum['neg']}
         mean pairs/anchor   = {train_sum['mean_pairs_per_anchor']:.2f}
         median pairs/anchor = {train_sum['median_pairs_per_anchor']:.2f}

Val   : {val_sum['pairs']} pairs  |  {val_sum['anchors']} anchors
         pos={val_sum['pos']}  neg={val_sum['neg']}
         mean pairs/anchor   = {val_sum['mean_pairs_per_anchor']:.2f}
         median pairs/anchor = {val_sum['median_pairs_per_anchor']:.2f}

Anchor overlap (train \u2229 val) : {len(anchor_overlap)}
Negatives per split :
  train: hard={train_stats['neg_hard']}  cross={train_stats['neg_cross']}  reduced_anchors={train_stats['reduced']}
  val  : hard={val_stats['neg_hard']}  cross={val_stats['neg_cross']}  reduced_anchors={val_stats['reduced']}

Balancing strategy
  - max {MAX_POS_PER_ANCHOR} positives per anchor (random sub-sample if more)
  - n_neg = n_pos  (record's hard negatives first, then cross-anchor fill)
  - cross-anchor pool restricted to positives of OTHER anchors within the same split

Files
  - train.json
  - val.json
  - DATASET_CARD.txt
================================================================
"""

with open(OUTPUT_DIR / "DATASET_CARD.txt", "w", encoding="utf-8") as f:
    f.write(card)

print(card)
print(f"\n\u2705 Written: {OUTPUT_DIR}")


  Qwen Binary-Classifier Dataset  —  WIKIDATA
Format       : anchor_query | query | label (0 or 1)
Split source : splits_wikidata.json  (train_idx / valid_idx)
Raw source   : jaccard_triplets_wikidata.json

Train : 97138 pairs  |  8588 anchors
         pos=48569  neg=48569
         mean pairs/anchor   = 11.31
         median pairs/anchor = 4.00

Val   : 12366 pairs  |  1073 anchors
         pos=6183  neg=6183
         mean pairs/anchor   = 11.52
         median pairs/anchor = 4.00

Anchor overlap (train ∩ val) : 0
Negatives per split :
  train: hard=48569  cross=0  reduced_anchors=0
  val  : hard=6183  cross=0  reduced_anchors=0

Balancing strategy
  - max 20 positives per anchor (random sub-sample if more)
  - n_neg = n_pos  (record's hard negatives first, then cross-anchor fill)
  - cross-anchor pool restricted to positives of OTHER anchors within the same split

Files
  - train.json
  - val.json
  - DATASET_CARD.txt


✅ Written: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Em